# Glitch × ShortGPT — remote video studio

This notebook runs the open-source ShortGPT video engine in a **Google Colab cloud runtime**. Nothing is installed on your PC. Keep this notebook tab open while you use the Video Studio in Glitch.

## Free setup
- A Google AI Studio API key (use the free Gemini tier where available): https://aistudio.google.com/app/apikey
- A free Pexels API key for stock footage: https://www.pexels.com/api/
- Choose **Edge TTS** in ShortGPT for free voice generation. Do not select ElevenLabs unless you independently have a key.

ShortGPT's Colab runtime is temporary and can disconnect when idle or when Colab reclaims it. Restart this notebook when that happens; the share URL may change.


In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg imagemagick
!sed -i '/<policy domain="path" rights="none" pattern="@\*"/d' /etc/ImageMagick-6/policy.xml 2>/dev/null || true
!ffmpeg -version | head -n 1


In [ ]:
import os, pathlib, subprocess, sys
REPO = "/content/ShortGPT"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "stable", "https://github.com/RayVentura/ShortGPT.git", REPO], check=True)
os.chdir(REPO)
packages = ["gradio==5.12.0", "gradio_client==1.5.4", "openai==1.37.0", "httpx==0.27.2", "tiktoken", "tinydb", "tinymongo", "proglog", "yt-dlp>=2025.1.12", "whisper-timestamped", "protobuf==3.20.3", "pillow==10.4.0", "moviepy==2.1.2", "progress", "questionary", "edge-tts"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("ShortGPT dependencies installed. Colab's preinstalled PyTorch/Torchaudio are being reused.")


In [ ]:
from getpass import getpass
from shortGPT.config.api_db import ApiKeyManager, ApiProvider

gemini_key = getpass("Google AI Studio Gemini API key (input hidden): ").strip()
pexels_key = getpass("Pexels API key (input hidden): ").strip()
if not gemini_key or not pexels_key:
    raise ValueError("Both Gemini and Pexels keys are required. Use the free-key links in the first cell.")
ApiKeyManager.set_api_key(ApiProvider.GEMINI, gemini_key)
ApiKeyManager.set_api_key(ApiProvider.PEXELS, pexels_key)
os.environ["GEMINI_API_KEY"] = gemini_key
os.environ["PEXELS_API_KEY"] = pexels_key
del gemini_key, pexels_key
print("Keys stored in this temporary Colab runtime. They are not sent to Glitch.")


In [ ]:
from gui.gui_gradio import ShortGptUI
print("Starting ShortGPT. Wait for the public https://....gradio.live link below.")
app = ShortGptUI(colab=True)
app.launch()
